# MODEL BUILDING
After completing feature engineering and preprocessing, the next step is to train machine learning models to predict the Air Quality Index (AQI).
The objective is to compare multiple regression algorithms and identify the model that predicts AQI with the highest accuracy on unseen data.

In [2]:
import numpy as np
import pandas as pd

air_df = pd.read_csv("../data/air_quality_feature_engineered.csv")
air_df.head()

,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,AQI,Year,Month,Day,City_Encoded,Season_Encoded,DayOfWeek_Encoded
0,83.13,96.18,6.93,28.71,33.72,16.31,6.93,49.52,59.76,0.02,0.00,209.0,2015,1,29,0,3,4
1,79.84,96.18,13.85,28.68,41.08,16.31,13.85,48.49,97.07,0.04,0.00,328.0,2015,1,30,0,3,0
2,94.52,96.18,24.39,32.66,52.61,16.31,24.39,67.39,111.33,0.24,0.01,514.0,2015,1,31,0,3,2
3,135.99,96.18,43.48,42.08,84.57,16.31,43.48,75.23,102.70,0.40,0.04,782.0,2015,2,1,0,3,3
4,178.33,96.18,54.56,35.31,72.80,16.31,54.56,55.04,107.38,0.46,0.06,914.0,2015,2,2,0,3,1


In [3]:
X = air_df.drop("AQI", axis=1)
y = air_df["AQI"]

print("Feature Shape :", X.shape)
print("Target Shape  :", y.shape)

Feature Shape : (24850, 17)
Target Shape  : (24850,)


In [4]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Feature scaling completed.")

Feature scaling completed.


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Features :", X_train.shape)
print("Testing Features  :", X_test.shape)
print("Training Labels   :", y_train.shape)
print("Testing Labels    :", y_test.shape)

Training Features : (19880, 17)
Testing Features  : (4970, 17)
Training Labels   : (19880,)
Testing Labels    : (4970,)


In [7]:
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
print("Linear Regression model trained successfully.")

Linear Regression model trained successfully.


In [8]:
# Train Decision Tree Regressor

from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(
    random_state=42
)

dt_model.fit(X_train, y_train)

print("Decision Tree model trained successfully.")

Decision Tree model trained successfully.


In [6]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [9]:
# Train Gradient Boosting Regressor

from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    random_state=42
)

gb_model.fit(X_train, y_train)

print("Gradient Boosting model trained successfully.")

Gradient Boosting model trained successfully.


# Model_Evaluation

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# Evaluation Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [11]:
# Predictions from all models

lr_predictions = lr_model.predict(X_test)

dt_predictions = dt_model.predict(X_test)

rf_predictions = rf_model.predict(X_test)

gb_predictions = gb_model.predict(X_test)

print("Predictions generated successfully.")

Predictions generated successfully.


In [12]:
def evaluate_model(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {
        "Model": model_name,
        "MAE": round(mae, 2),
        "MSE": round(mse, 2),
        "RMSE": round(rmse, 2),
        "R² Score": round(r2, 4)
    }
print("Evaluation function created.")

Evaluation function created.


In [13]:
results = []

results.append(
    evaluate_model(
        "Linear Regression",
        y_test,
        lr_predictions
    )
)

results.append(
    evaluate_model(
        "Decision Tree",
        y_test,
        dt_predictions
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        y_test,
        rf_predictions
    )
)

results.append(
    evaluate_model(
        "Gradient Boosting",
        y_test,
        gb_predictions
    )
)

results_df = pd.DataFrame(results)

results_df

,Model,MAE,MSE,RMSE,R² Score
0,Linear Regression,31.05,3501.36,59.17,0.8088
1,Decision Tree,28.68,3337.31,57.77,0.8177
2,Random Forest,20.55,1637.03,40.46,0.9106
3,Gradient Boosting,23.36,1920.83,43.83,0.8951


## Model Performance Comparison

The models are sorted based on their R² Score to identify the best-performing regression algorithm for AQI prediction.

In [14]:
results_df = results_df.sort_values(
    by="R² Score",
    ascending=False
)
results_df.reset_index(drop=True)

,Model,MAE,MSE,RMSE,R² Score
0,Random Forest,20.55,1637.03,40.46,0.9106
1,Gradient Boosting,23.36,1920.83,43.83,0.8951
2,Decision Tree,28.68,3337.31,57.77,0.8177
3,Linear Regression,31.05,3501.36,59.17,0.8088


In [15]:
fig = px.bar(
    results_df,
    x="Model",
    y="R² Score",
    color="R² Score",
    text="R² Score",
    title="Comparison of R² Scores for AQI Prediction Models"
)

fig.update_traces(
    textposition="outside"
)

fig.show()

In [16]:
fig = px.bar(
    results_df,
    x="Model",
    y="RMSE",
    color="RMSE",
    text="RMSE",
    title="RMSE Comparison Across Regression Models"
)

fig.update_traces(
    textposition="outside"
)

fig.show()

In [17]:
prediction_df = pd.DataFrame({
    "Actual AQI": y_test,
    "Predicted AQI": rf_predictions
})

fig = px.scatter(
    prediction_df.sample(1500, random_state=42),
    x="Actual AQI",
    y="Predicted AQI",
    title="Actual vs Predicted AQI (Random Forest)"
)

fig.show()

In [18]:
prediction_df["Residual"] = (
    prediction_df["Actual AQI"] -
    prediction_df["Predicted AQI"]
)

fig = px.histogram(
    prediction_df,
    x="Residual",
    nbins=50,
    title="Residual Error Distribution"
)

fig.show()

In [19]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(10)

,Feature,Importance
0,PM2.5,0.489375
6,CO,0.366113
1,PM10,0.035985
2,NO,0.035651
8,O3,0.012399
4,NOx,0.009944
7,SO2,0.008046
10,Toluene,0.006918
3,NO2,0.006254
13,Day,0.005622


In [20]:
top_features = feature_importance.head(10)

fig = px.bar(
    top_features,
    x="Importance",
    y="Feature",
    orientation="h",
    color="Importance",
    title="Top 10 Features Influencing AQI Prediction"
)

fig.show()

In [21]:
best_model = results_df.iloc[0]

print("Best Model")
print(best_model)

Best Model
Model       Random Forest
MAE                 20.55
MSE               1637.03
RMSE                40.46
R² Score           0.9106
Name: 2, dtype: object


In [22]:
import joblib

joblib.dump(
    rf_model,
    "best_aqi_model.pkl"
)

joblib.dump(
    scaler,
    "aqi_scaler.pkl"
)

print("Model and scaler saved successfully.")

Model and scaler saved successfully.


In [23]:
import joblib
import sklearn

print("Scikit-learn Version:", sklearn.__version__)

joblib.dump(rf_model, "best_aqi_model.pkl")
joblib.dump(scaler, "aqi_scaler.pkl")

print("Model and scaler saved successfully!")

Scikit-learn Version: 1.7.1
Model and scaler saved successfully!


In [24]:
import sklearn

print("Scikit-learn version used for training:", sklearn.__version__)

Scikit-learn version used for training: 1.7.1


In [25]:
import joblib
import sklearn

print("Saving model with Scikit-learn:", sklearn.__version__)

joblib.dump(rf_model, "../models/best_aqi_model.pkl")
joblib.dump(scaler, "../models/aqi_scaler.pkl")

print("✅ Model and scaler saved successfully.")

Saving model with Scikit-learn: 1.7.1
✅ Model and scaler saved successfully.
